# Intervention Drift Analysis

This notebook loads preserved OQI selected-feature parity artifacts, summarizes retained-feature and tracked-logit drift, and inspects how the strongest layer-specific errors appear to propagate through retained feature rows.

Typical artifact generation flow:

```bash
cd /home/speediedan/repos/interpretune && \
  IT_RUN_STANDALONE_TESTS=1 \
  IT_PARITY_PRESERVE_ARTIFACTS=1 \
  IT_PARITY_PRESERVE_ARTIFACT_DIR=/home/speediedan/repos/interpretune/tests/parity_analysis/artifacts \
  /mnt/cache/speediedan/.venvs/it_latest/bin/python -m pytest \
  tests/core/test_analysis_backend_parity.py \
  -k 'oqi_debug_selected_feature_adjacency_trace' -vv
```

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "tests").exists():
            return candidate
    raise RuntimeError(f"Unable to locate repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.it_examples.utils.nb_ui_utils import display_layer_divergence_summary, display_logit_drift_summary
from src.it_examples.utils.raw_graph_analysis import plot_ridgeline_convergence
from tests.parity_analysis.intervention_drift_analysis import (
    PRESERVE_ARTIFACT_DIR_ENV,
    build_cli_summary,
    build_layer_error_distribution,
    load_preserved_intervention_artifacts,
)

In [2]:
artifact_root = Path(os.environ.get(PRESERVE_ARTIFACT_DIR_ENV, REPO_ROOT / "tests" / "parity_analysis" / "artifacts"))
artifact_dirs = sorted([path for path in artifact_root.glob("*") if path.is_dir()])
if not artifact_dirs:
    raise FileNotFoundError(f"No preserved parity artifacts found under {artifact_root}")
artifact_dir = artifact_dirs[-1]
artifact_dir

PosixPath('/home/speediedan/repos/interpretune/tests/parity_analysis/artifacts/gemma3_4b_it_oqi_gemma_3_4b_it__25_gemmascope_2_transcoder_16k__60_20260412_005225')

In [3]:
artifacts = load_preserved_intervention_artifacts(artifact_dir)
summary = build_cli_summary(artifacts, target_layer=33, top_k_rows=5, top_k_sources=5)
summary["feature_row"], summary["report"]["layer_with_max_divergence"]

([25, 34, 60], 33)

In [4]:
display_layer_divergence_summary(summary["report"]["layer_summaries"])
display_logit_drift_summary(summary["report"]["logit_summary"])
summary["report"]["top_feature_errors"][:10]

Layer,Diverged,Total,Max |Error|,Mean |Error|,Σ|Expected Δ|,Σ|Actual Δ|,Top Error Row
33,0,32,0.0013,3.42e-04,3764.7051,3764.7070,"(33, 34, 1966)"
28,0,135,8.54e-04,8.40e-05,11931.1074,11931.1113,"(28, 34, 2906)"
26,0,105,7.93e-04,6.40e-05,6749.1680,6749.1709,"(26, 34, 6885)"
27,0,135,7.93e-04,6.92e-05,7113.4658,7113.4707,"(27, 34, 15335)"
31,0,117,6.41e-04,1.17e-04,6291.1147,6291.1143,"(31, 34, 8278)"
32,0,83,6.10e-04,9.45e-05,4010.3772,4010.3779,"(32, 34, 6329)"
30,0,125,5.80e-04,7.37e-05,6647.1299,6647.1313,"(30, 34, 7123)"
29,0,155,5.34e-04,7.56e-05,8544.7793,8544.7783,"(29, 34, 8292)"
0,0,262,0.00e+00,0.00e+00,0.00e+00,0.00e+00,"(0, 4, 194)"
1,0,319,0.00e+00,0.00e+00,0.00e+00,0.00e+00,"(1, 4, 125)"


Token,Token ID,Expected Δ,Actual Δ,|Error|
Houston,110654,9.0407,9.0407,3.81e-05
Texas,69033,2.0122,2.0123,2.29e-05
Austin,107305,7.0131,7.0131,1.14e-05
Dallas,120384,9.3630,9.3630,7.63e-06
ramento,36580,2.7862,2.7862,5.72e-06
California,52247,2.0127,2.0127,3.81e-06


[{'feature_row': [33, 34, 1966],
  'node_index': 8175,
  'baseline_activation': 317.4001159667969,
  'expected_activation': 367.41632080078125,
  'actual_activation': 367.4175720214844,
  'expected_delta': 50.016197204589844,
  'actual_delta': 50.0174560546875,
  'abs_error': 0.001251220703125,
  'diverged': False,
  'sign_mismatch': False},
 {'feature_row': [33, 34, 9188],
  'node_index': 8188,
  'baseline_activation': 779.1107788085938,
  'expected_activation': 791.5225219726562,
  'actual_activation': 791.523681640625,
  'expected_delta': 12.411722183227539,
  'actual_delta': 12.41290283203125,
  'abs_error': 0.00115966796875,
  'diverged': False,
  'sign_mismatch': False},
 {'feature_row': [33, 34, 1955],
  'node_index': 8174,
  'baseline_activation': 144.35472106933594,
  'expected_activation': 105.1199951171875,
  'actual_activation': 105.12098693847656,
  'expected_delta': -39.2347297668457,
  'actual_delta': -39.233734130859375,
  'abs_error': 0.0009918212890625,
  'diverged': 

In [5]:
layer_error_distribution = build_layer_error_distribution(artifacts.report)
plot_ridgeline_convergence(
    layer_error_distribution,
    title="Absolute retained-feature activation error distribution by layer",
)

In [6]:
summary["propagation"]

[{'target_feature_row': [33, 34, 1966],
  'target_node_index': 8175,
  'target_abs_error': 0.001251220703125,
  'target_actual_delta': 50.0174560546875,
  'target_expected_delta': 50.016197204589844,
  'incoming_sources': [{'source_feature_row': [32, 34, 7063],
    'source_node_index': 8136,
    'source_abs_error': 0.0003662109375,
    'source_actual_delta': 27.444580078125,
    'source_expected_delta': 27.44490623474121,
    'edge_weight': 183.56524658203125},
   {'source_feature_row': [29, 34, 842],
    'source_node_index': 7768,
    'source_abs_error': 0.000244140625,
    'source_actual_delta': 103.2247314453125,
    'source_expected_delta': 103.22498321533203,
    'edge_weight': -159.5984649658203},
   {'source_feature_row': [31, 34, 12196],
    'source_node_index': 8060,
    'source_abs_error': 0.00048828125,
    'source_actual_delta': 176.6591796875,
    'source_expected_delta': 176.65956115722656,
    'edge_weight': 159.3763427734375},
   {'source_feature_row': [31, 34, 3634],
 

In [7]:
print(json.dumps(summary, indent=2))

{
  "artifact_dir": "/home/speediedan/repos/interpretune/tests/parity_analysis/artifacts/gemma3_4b_it_oqi_gemma_3_4b_it__25_gemmascope_2_transcoder_16k__60_20260412_005225",
  "feature_row": [
    25,
    34,
    60
  ],
  "metadata": {
    "constrained_feature_ref": "gemma-3-4b-it/25-gemmascope-2-transcoder-16k/60",
    "feature_row": [
      25,
      34,
      60
    ],
    "prompt_render_mode": "apply_chat_template",
    "requested_key_tokens": [
      "Sacramento",
      "Austin",
      "Dallas",
      "California",
      "Houston",
      "Texas"
    ],
    "graph_target_ids": [
      36580,
      107305,
      120384,
      52247,
      110654,
      69033
    ],
    "graph_target_tokens": [
      "ramento",
      "Austin",
      "Dallas",
      "California",
      "Houston",
      "Texas"
    ],
    "rendered_prompt": "<bos><start_of_turn>user\nAnswer with only the missing city name. Fact: the capital of the only US state with greater population than the state containing Dallas 